In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
import requests
resp = requests.get('https://httpbin.org/get', timeout=10)
print(resp.status_code)

200


In [3]:
url = 'https://github.com/DataTalksClub/faq/archive/refs/heads/main.zip'
resp = requests.get(url, timeout=30)
print(resp.status_code)

200


In [4]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [5]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
print(dtc_faq[0])

{'description': 'Review and process open FAQ PRs', 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep -r

In [6]:
evidently_docs = read_repo_data('evidentlyai', 'docs')

In [7]:
print(f"Evidently docs: {len(evidently_docs)}")

Evidently docs: 95


In [8]:
doc = evidently_docs[45]
print(doc.keys())
print(f"Content length: {len(doc['content'])} characters")

dict_keys(['title', 'description', 'content', 'filename'])
Content length: 21712 characters


In [9]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [10]:
test = sliding_window(evidently_docs[45]['content'], 2000, 1000)
print(f"Number of chunks: {len(test)}")
print(f"First chunk start: {test[0]['start']}")
print(f"First chunk length: {len(test[0]['chunk'])}")

Number of chunks: 21
First chunk start: 0
First chunk length: 2000


In [12]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [13]:
text = evidently_docs[45]['content']
sections = split_markdown_by_level(text, level=2)
print(f"Number of sections: {len(sections)}")
print(sections[0])

Number of sections: 8
## 1. Installation and Imports

Install Evidently:

```python
pip install evidently[llm] 
```

Import the required modules:

```python
import pandas as pd
from evidently.future.datasets import Dataset
from evidently.future.datasets import DataDefinition
from evidently.future.datasets import Descriptor
from evidently.future.descriptors import *
from evidently.future.report import Report
from evidently.future.presets import TextEvals
from evidently.future.metrics import *
from evidently.future.tests import *

from evidently.features.llm_judge import BinaryClassificationPromptTemplate
```

To connect to Evidently Cloud:

```python
from evidently.ui.workspace.cloud import CloudWorkspace
```

**Optional.** To create monitoring panels as code:

```python
from evidently.ui.dashboards import DashboardPanelPlot
from evidently.ui.dashboards import DashboardPanelTestSuite
from evidently.ui.dashboards import DashboardPanelTestSuiteCounter
from evidently.ui.dashboards import T

In [14]:
evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

In [15]:
print(f"Total chunks: {len(evidently_chunks)}")
print(evidently_chunks[0].keys())

Total chunks: 266
dict_keys(['title', 'description', 'filename', 'section'])


In [23]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def llm(prompt, model='llama-3.3-70b-versatile'):
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=messages
    )
    
    return response.choices[0].message.content

In [18]:
print(llm("Say hello in one sentence."))

Hello, how are you today?


In [25]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

IMPORTANT: You MUST separate each section with --- on its own line. Do not skip this.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [26]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [27]:
test_sections = intelligent_chunking(evidently_docs[45]['content'])
print(f"Number of sections: {len(test_sections)}")
print(test_sections[0])

Number of sections: 9
## Introduction to Regression Testing for LLM Outputs

In this tutorial, you will learn how to perform regression testing for LLM outputs. You can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.


In [28]:
from tqdm.auto import tqdm

evidently_chunks_llm = []

for doc in tqdm(evidently_docs[:5]):  # just first 5 docs
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks_llm.append(section_doc)

print(f"Total chunks: {len(evidently_chunks_llm)}")

  0%|          | 0/5 [00:00<?, ?it/s]

Total chunks: 21
